In [2]:
pip install geopandas matplotlib requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 5.9 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import requests
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
ACS_PATH   = "/Users/aliabonanno/public_policy_project/acs_poverty_panel.csv"
NOAA_PATH  = "/Users/aliabonanno/public_policy_project/noaa_heat_panel.csv"
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading data...")
acs  = pd.read_csv(ACS_PATH,  dtype={"FIPS": str})
noaa = pd.read_csv(NOAA_PATH, dtype={"FIPS": str})

# Pad FIPS just in case
acs["FIPS"]  = acs["FIPS"].str.zfill(5)
noaa["FIPS"] = noaa["FIPS"].str.zfill(5)

# ── MERGE ────────────────────────────────────────────────────────────────────
# NOAA is monthly, ACS is annual — merge on FIPS + year
panel = noaa.merge(acs[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
                   on=["FIPS", "year"], how="inner")

print(f"Merged panel: {len(panel)} rows")
print(f"Counties: {panel['FIPS'].nunique()}, Years: {sorted(panel['year'].unique())}")
print(f"Missing poverty: {panel['pct_below_150pct_poverty'].isna().sum()}")

# Save merged panel
panel.to_csv("/Users/aliabonanno/public_policy_project/merged_panel.csv", index=False)
print("Saved merged_panel.csv")

# ── TABLE 1: SUMMARY STATISTICS ───────────────────────────────────────────────
print("\n--- TABLE 1 ---")
summer = panel[panel["month"].isin([6, 7, 8])]

stats_vars = {
    "Poverty rate (% below 150% FPL)": "pct_below_150pct_poverty",
    "Days above 90°F (summer months)": "days_above_90f",
    "Days above 95°F (summer months)": "days_above_95f",
    "Days above 100°F (summer months)": "days_above_100f",
}

rows = []
for label, col in stats_vars.items():
    src = summer if "summer" in label else panel
    rows.append({
        "Variable": label,
        "Mean":  src[col].mean(),
        "SD":    src[col].std(),
        "Min":   src[col].min(),
        "P25":   src[col].quantile(0.25),
        "P75":   src[col].quantile(0.75),
        "Max":   src[col].max(),
        "N":     src[col].count(),
    })

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False, float_format="{:.3f}".format))

# Save as CSV for easy copying
table1.to_csv(os.path.join(OUTPUT_DIR, "table1_summary_stats.csv"), index=False)

# ── COUNTY-LEVEL AVERAGES FOR MAPS ───────────────────────────────────────────
# Average over all years and summer months
summer_avg = (summer.groupby("FIPS")
              .agg(avg_days_95f=("days_above_95f", "mean"),
                   avg_poverty=("pct_below_150pct_poverty", "mean"),
                   avg_pop=("total_pop", "mean"))
              .reset_index())

# ── DOWNLOAD COUNTY SHAPEFILE IF NEEDED ──────────────────────────────────────
shp_path = "/Users/aliabonanno/public_policy_project/counties.zip"
if not os.path.exists(shp_path):
    print("\nDownloading county shapefile...")
    url = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_5m.zip"
    r = requests.get(url, timeout=60)
    with open(shp_path, "wb") as f:
        f.write(r.content)
    print("Downloaded.")

gdf = gpd.read_file(f"zip://{shp_path}")
gdf = gdf.rename(columns={"GEOID": "FIPS"})

# Drop Alaska, Hawaii, territories for cleaner map
gdf = gdf[~gdf["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])]

# Merge spatial data
gdf = gdf.merge(summer_avg, on="FIPS", how="left")

# ── FIGURE 1: HEAT MAP ────────────────────────────────────────────────────────
print("\nGenerating Figure 1: Heat map...")
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
gdf.plot(column="avg_days_95f",
         ax=ax,
         cmap="YlOrRd",
         legend=True,
         legend_kwds={"label": "Avg. days above 95°F (June–Aug, 2012–2024)",
                      "orientation": "horizontal", "shrink": 0.6},
         missing_kwds={"color": "lightgrey"})
ax.set_title("Average Summer Days Above 95°F by County (2012–2024)",
             fontsize=14, fontweight="bold", pad=12)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig1_heat_map.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig1_heat_map.png")

# ── FIGURE 2: POVERTY MAP ─────────────────────────────────────────────────────
print("Generating Figure 2: Poverty map...")
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
gdf.plot(column="avg_poverty",
         ax=ax,
         cmap="Blues",
         legend=True,
         legend_kwds={"label": "Share of population below 150% FPL (2012–2024)",
                      "orientation": "horizontal", "shrink": 0.6},
         missing_kwds={"color": "lightgrey"})
ax.set_title("Average County Poverty Rate (% below 150% FPL, 2012–2024)",
             fontsize=14, fontweight="bold", pad=12)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig2_poverty_map.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig2_poverty_map.png")

# ── FIGURE 3: BINNED SCATTER ──────────────────────────────────────────────────
print("Generating Figure 3: Binned scatter...")

# Use summer county-year observations
# Split into high/low poverty at median
median_pov = summer["pct_below_150pct_poverty"].median()
summer = summer.copy()
summer["poverty_group"] = np.where(
    summer["pct_below_150pct_poverty"] >= median_pov,
    "High poverty (above median)",
    "Low poverty (below median)"
)

# Create heat bins: 0, 1-3, 4-7, 8-12, 13-20, 21+ days above 95F
bins   = [-1, 0, 3, 7, 12, 20, 31]
labels = ["0", "1–3", "4–7", "8–12", "13–20", "21+"]
summer["heat_bin"] = pd.cut(summer["days_above_95f"], bins=bins, labels=labels)

# Mean poverty rate by heat bin and poverty group
# (showing how poverty distributes across heat exposure)
binned = (summer.groupby(["heat_bin", "poverty_group"], observed=True)
          ["pct_below_150pct_poverty"]
          .mean()
          .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
for group, color in [("High poverty (above median)", "#d73027"),
                     ("Low poverty (below median)",  "#4575b4")]:
    sub = binned[binned["poverty_group"] == group]
    ax.plot(sub["heat_bin"].astype(str),
            sub["pct_below_150pct_poverty"],
            marker="o", linewidth=2, markersize=7,
            label=group, color=color)

ax.set_xlabel("Days above 95°F in month", fontsize=12)
ax.set_ylabel("Mean poverty rate (% below 150% FPL)", fontsize=12)
ax.set_title("Poverty Rate Across Heat Exposure Bins\n(County-month observations, summers 2012–2024)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3_binned_scatter.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig3_binned_scatter.png")

print(f"\nAll figures saved to {OUTPUT_DIR}")
print("Files produced:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")

Loading data...
Merged panel: 248832 rows
Counties: 1597, Years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Missing poverty: 0
Saved merged_panel.csv

--- TABLE 1 ---
                        Variable  Mean    SD   Min   P25    P75    Max      N
 Poverty rate (% below 150% FPL) 0.263 0.088 0.012 0.199  0.317  0.671 248832
 Days above 90°F (summer months) 7.784 8.801 0.000 0.000 13.000 31.000  62208
 Days above 95°F (summer months) 1.998 4.550 0.000 0.000  2.000 31.000  62208
Days above 100°F (summer months) 0.371 1.988 0.000 0.000  0.000 31.000  62208

Downloaded.

Generating Figure 1: Heat map...
Saved fig1_heat_map.png
Generating Figure 2: Poverty map...
Saved fig2_poverty_map.png
Generating Figure 3: Binned scatter...
Saved fig3_binned_scatter.png

All figures saved to /Users/aliabonanno/public_policy_project/figures
Files produced:
  fig3_binned_scatter.png
  fig1_heat_map.png
  fig2_poverty_map.png
  table1_summary_stats.csv


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
MERGED_PATH = "/Users/aliabonanno/public_policy_project/merged_panel.csv"
OUTPUT_DIR  = "/Users/aliabonanno/public_policy_project/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LOAD ─────────────────────────────────────────────────────────────────────
panel = pd.read_csv(MERGED_PATH, dtype={"FIPS": str})
summer = panel[panel["month"].isin([6, 7, 8])].copy()

# ── FIGURE 3A: HEAT DAYS BY POVERTY QUARTILE (BAR CHART) ─────────────────────
# Assign poverty quartile based on county average poverty rate
county_avg_pov = (summer.groupby("FIPS")["pct_below_150pct_poverty"]
                  .mean().reset_index()
                  .rename(columns={"pct_below_150pct_poverty": "avg_poverty"}))

county_avg_pov["poverty_quartile"] = pd.qcut(
    county_avg_pov["avg_poverty"], q=4,
    labels=["Q1\n(Lowest poverty)", "Q2", "Q3", "Q4\n(Highest poverty)"]
)

summer = summer.merge(county_avg_pov[["FIPS", "poverty_quartile"]], on="FIPS", how="left")

quartile_heat = (summer.groupby("poverty_quartile", observed=True)
                 [["days_above_90f", "days_above_95f", "days_above_100f"]]
                 .mean().reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(quartile_heat))
width = 0.25

bars1 = ax.bar(x - width, quartile_heat["days_above_90f"],
               width, label="Days above 90°F", color="#fdae61", edgecolor="white")
bars2 = ax.bar(x,         quartile_heat["days_above_95f"],
               width, label="Days above 95°F", color="#d73027", edgecolor="white")
bars3 = ax.bar(x + width, quartile_heat["days_above_100f"],
               width, label="Days above 100°F", color="#7f0000", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(quartile_heat["poverty_quartile"].astype(str), fontsize=11)
ax.set_xlabel("County poverty quartile (% below 150% FPL)", fontsize=12)
ax.set_ylabel("Mean days above threshold (June–Aug)", fontsize=12)
ax.set_title("Summer Heat Exposure by County Poverty Quartile\n(2012–2024)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3a_heat_by_poverty_quartile.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig3a_heat_by_poverty_quartile.png")

# ── FIGURE 3B: SCATTER - COUNTY AVG POVERTY VS AVG HEAT DAYS ─────────────────
county_heat = (summer.groupby("FIPS")
               .agg(avg_days_95f=("days_above_95f", "mean"),
                    avg_days_90f=("days_above_90f", "mean"),
                    avg_poverty=("pct_below_150pct_poverty", "mean"),
                    avg_pop=("total_pop", "mean"))
               .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(county_heat["avg_poverty"],
                county_heat["avg_days_95f"],
                c=county_heat["avg_days_95f"],
                cmap="YlOrRd", alpha=0.5, s=15,
                linewidths=0)

# Add a simple linear trend line
z = np.polyfit(county_heat["avg_poverty"].dropna(),
               county_heat["avg_days_95f"].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(county_heat["avg_poverty"].min(),
                     county_heat["avg_poverty"].max(), 100)
ax.plot(x_line, p(x_line), color="#d73027", linewidth=2,
        linestyle="--", label=f"Linear trend")

ax.set_xlabel("Average poverty rate (% below 150% FPL)", fontsize=12)
ax.set_ylabel("Average days above 95°F (June–Aug)", fontsize=12)
ax.set_title("County-Level Poverty Rate vs. Summer Heat Exposure\n(2012–2024 averages)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Annotate correlation
corr = county_heat[["avg_poverty", "avg_days_95f"]].corr().iloc[0, 1]
ax.annotate(f"r = {corr:.3f}", xy=(0.05, 0.92), xycoords="axes fraction",
            fontsize=11, color="#333333")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3b_scatter_poverty_heat.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig3b_scatter_poverty_heat.png")

# ── PRINT QUARTILE TABLE ──────────────────────────────────────────────────────
print("\nMean heat days by poverty quartile:")
print(quartile_heat.to_string(index=False, float_format="{:.2f}".format))

print(f"\nCorrelation between avg poverty and avg days above 95F: {corr:.3f}")

Saved fig3a_heat_by_poverty_quartile.png
Saved fig3b_scatter_poverty_heat.png

Mean heat days by poverty quartile:
     poverty_quartile  days_above_90f  days_above_95f  days_above_100f
 Q1\n(Lowest poverty)            8.00            1.89             0.28
                   Q2            7.70            2.07             0.51
                   Q3            8.44            2.36             0.46
Q4\n(Highest poverty)            6.99            1.67             0.24

Correlation between avg poverty and avg days above 95F: -0.024


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
MERGED_PATH = "/Users/aliabonanno/public_policy_project/merged_panel.csv"
OUTPUT_DIR  = "/Users/aliabonanno/public_policy_project/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

panel = pd.read_csv(MERGED_PATH, dtype={"FIPS": str})
summer = panel[panel["month"].isin([6, 7, 8])].copy()

# Annual summer heat days per county (sum across June-Aug)
annual_heat = (summer.groupby(["FIPS", "year", "pct_below_150pct_poverty"])
               ["days_above_95f"].sum().reset_index()
               .rename(columns={"days_above_95f": "summer_days_95f"}))

# ── FIGURE A: WITHIN-COUNTY VARIATION FOR SELECTED COUNTIES ──────────────────
# Pick 6 illustrative counties spanning different poverty levels and regions
example_counties = {
    "01001": "Autauga Co., AL\n(South, mid-poverty)",
    "48113": "Dallas Co., TX\n(South, low-poverty)",
    "28049": "Hinds Co., MS\n(South, high-poverty)",
    "04013": "Maricopa Co., AZ\n(Southwest, low-poverty)",
    "37119": "Mecklenburg Co., NC\n(South, low-poverty)",
    "21013": "Bell Co., KY\n(Appalachia, high-poverty)",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes = axes.flatten()

for i, (fips, label) in enumerate(example_counties.items()):
    ax = axes[i]
    sub = annual_heat[annual_heat["FIPS"] == fips].sort_values("year")

    if len(sub) == 0:
        ax.set_title(f"{label}\n(no data)", fontsize=9)
        continue

    pov = sub["pct_below_150pct_poverty"].mean()
    mean_heat = sub["summer_days_95f"].mean()

    ax.bar(sub["year"], sub["summer_days_95f"],
           color="#d73027", alpha=0.75, edgecolor="white", width=0.7)
    ax.axhline(mean_heat, color="black", linewidth=1.5,
               linestyle="--", label=f"Mean: {mean_heat:.1f} days")

    ax.set_title(f"{label}\nPoverty rate: {pov:.1%}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Year", fontsize=8)
    ax.set_ylabel("Days above 95°F\n(Jun–Aug)", fontsize=8)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8)
    ax.set_xticks(sub["year"].values[::2])
    ax.set_xticklabels(sub["year"].values[::2], rotation=45)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Year-to-Year Variation in Summer Heat Days Within Counties\n"
             "(Identifying variation used in regression)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4a_within_county_variation.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig4a_within_county_variation.png")

# ── FIGURE B: DISTRIBUTION OF WITHIN-COUNTY SD IN HEAT DAYS ─────────────────
# Show that there IS meaningful within-county variation to identify off of
within_sd = (annual_heat.groupby("FIPS")["summer_days_95f"]
             .std().reset_index()
             .rename(columns={"summer_days_95f": "sd_heat"}))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(within_sd["sd_heat"].dropna(), bins=40,
        color="#d73027", alpha=0.75, edgecolor="white")
ax.axvline(within_sd["sd_heat"].median(), color="black",
           linewidth=2, linestyle="--",
           label=f"Median SD: {within_sd['sd_heat'].median():.1f} days")
ax.set_xlabel("Within-county SD in summer days above 95°F (across years)", fontsize=12)
ax.set_ylabel("Number of counties", fontsize=12)
ax.set_title("Distribution of Year-to-Year Heat Variation Across Counties\n"
             "(Each county contributes 13 years of heat shocks)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4b_within_county_sd.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig4b_within_county_sd.png")

# ── PRINT SUMMARY ─────────────────────────────────────────────────────────────
print(f"\nWithin-county SD in summer days above 95F:")
print(within_sd["sd_heat"].describe().round(2))

print(f"\nShare of counties with SD > 2 days: "
      f"{(within_sd['sd_heat'] > 2).mean():.1%}")
print(f"Share of counties with SD > 5 days: "
      f"{(within_sd['sd_heat'] > 5).mean():.1%}")

Saved fig4a_within_county_variation.png
Saved fig4b_within_county_sd.png

Within-county SD in summer days above 95F:
count    1597.00
mean        4.52
std         4.31
min         0.00
25%         1.11
50%         3.02
75%         7.15
max        19.16
Name: sd_heat, dtype: float64

Share of counties with SD > 2 days: 62.6%
Share of counties with SD > 5 days: 36.3%
